# VY CMa Spectrum — Herschel/HIFI + Planck 857 GHz

Observations from Herschel HIFI (HifiSScanModeDBS), combined across 7 frequency bands (714–1250 GHz), rebinned to constant resolving power R=250, and overlaid on a Planck 857 GHz continuum map.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.constants import c
from scipy.ndimage import gaussian_filter1d
from astropy.io import fits

## Observation metadata

Set `DATA_DIR` to the folder containing all your observation subdirectories.

In [ ]:
import os

# ── Edit this path ──────────────────────────────────────────────────────────
DATA_DIR = r"C:/Users/ellie/OneDrive/Desktop/Vy CMa"
# ────────────────────────────────────────────────────────────────────────────

def obs_path(obs_id, suffix):
    """Return the full path to a FITS file inside an observation folder.
    
    suffix should be the filename without the leading directory, e.g.
    'hhifiwbshssb_<obs_id>_red.fits'  — adjust as needed for your filenames.
    """
    return os.path.join(DATA_DIR, obs_id, suffix)

# ── Observation 1 ────────────────────────────────────────────────────────────
obs_ID_1 = "1342228611"
RA_1  = "07h 22m 58.32s"
Dec_1 = "-25d 46' 03.37''"
freq_range_min_1 = 1108.0610
freq_range_max_1 = 1243.9240

# ── Observation 2 ────────────────────────────────────────────────────────────
obs_ID_2 = "1342244512"
RA_2  = "07h 22m 58.30s"
Dec_2 = "-25d 46' 03.42''"
freq_range_min_2 = 1046.5830
freq_range_max_2 = 1121.9725

# ── Observation 3 ────────────────────────────────────────────────────────────
obs_ID_3 = "1342244631"
RA_3  = "07h 22m 58.34s"
Dec_3 = "-25d 46' 02.59''"
freq_range_min_3 = 799.0490
freq_range_max_3 = 859.9940

# ── Observation 4 ────────────────────────────────────────────────────────────
obs_ID_4 = "1342244789"
RA_4  = "07h 22m 58.32s"
Dec_4 = "-25d 46' 03.09''"
freq_range_min_4 = 949.0685
freq_range_max_4 = 1060.9755

# ── Observation 5 ────────────────────────────────────────────────────────────
obs_ID_5 = "1342244945"
RA_5  = "07h 22m 58.33s"
Dec_5 = "-25d 46' 03.51''"
freq_range_min_5 = 858.0570
freq_range_max_5 = 960.9905

# ── Observation 6 ────────────────────────────────────────────────────────────
obs_ID_6 = "1342244960"
RA_6  = "07h 22m 58.33s"
Dec_6 = "-25d 46' 02.92''"
freq_range_min_6 = 714.0385
freq_range_max_6 = 801.9965

# ── Observation 7 ────────────────────────────────────────────────────────────
obs_ID_7 = "1342244962"
RA_7  = "07h 22m 58.32s"
Dec_7 = "-25d 46' 03.12''"
freq_range_min_7 = 1227.1095
freq_range_max_7 = 1279.9600

## Load HIFI WBS FITS files

Each observation has a horizontal (H) and vertical (V) polarisation file.

> **Edit** the `h_file` / `v_file` strings below if your filenames differ — only the part after the obs_id folder matters.

In [ ]:
def load_hifi_wbs(obs_id, h_filename, v_filename):
    """Open H and V WBS FITS files and return (freq_h, flux_h, freq_v, flux_v)."""
    h = fits.open(os.path.join(DATA_DIR, obs_id, h_filename))
    v = fits.open(os.path.join(DATA_DIR, obs_id, v_filename))
    freq_h = h[1].data['frequency']
    flux_h = h[1].data['flux']
    freq_v = v[1].data['frequency']
    flux_v = v[1].data['flux']
    h.close(); v.close()
    return freq_h, flux_h, freq_v, flux_v

# ── Adjust filenames to match what is on disk ─────────────────────────────
# The original notebook had truncated paths; fill in the actual filenames.
# Pattern observed: hhifiwbshssb<...>.fits  /  hhifiwbsvssb<...>.fits

def wbs_filenames(obs_id):
    """Return (h_file, v_file) by globbing for the standard HIFI WBS naming pattern."""
    import glob
    folder = os.path.join(DATA_DIR, obs_id)
    h_matches = glob.glob(os.path.join(folder, "hhifiwbshssb*.fits"))
    v_matches = glob.glob(os.path.join(folder, "hhifiwbsvssb*.fits"))
    if not h_matches:
        raise FileNotFoundError(f"No H-pol WBS file found in {folder}")
    if not v_matches:
        raise FileNotFoundError(f"No V-pol WBS file found in {folder}")
    return os.path.basename(h_matches[0]), os.path.basename(v_matches[0])

obs_ids_ordered = [obs_ID_1, obs_ID_2, obs_ID_3, obs_ID_4, obs_ID_5, obs_ID_6, obs_ID_7]

freq_parts, flux_parts = [], []
for obs_id in obs_ids_ordered:
    h_file, v_file = wbs_filenames(obs_id)
    freq_h, flux_h, freq_v, flux_v = load_hifi_wbs(obs_id, h_file, v_file)
    freq_parts.extend([freq_h, freq_v])
    flux_parts.extend([flux_h, flux_v])
    print(f"Loaded obs {obs_id}: H={h_file}, V={v_file}")

freq_all = np.concatenate(freq_parts)
flux_all = np.concatenate(flux_parts)

order    = np.argsort(freq_all)
freq_all = freq_all[order]
flux_all = flux_all[order]

print(f"\nTotal samples: {len(freq_all):,}")
print(f"Frequency range: {freq_all.min():.3f} – {freq_all.max():.3f} GHz")

## Rebin to constant resolving power R = 250

In [ ]:
R = 250
r = np.exp(1.0 / R)  # multiplicative step per bin

edges = [freq_all.min()]
while edges[-1] < freq_all.max():
    edges.append(edges[-1] * r)
edges   = np.array(edges)
centers = np.sqrt(edges[:-1] * edges[1:])

inds = np.digitize(freq_all, edges) - 1
biny = np.full(centers.size, np.nan)
for i in range(centers.size):
    sel = inds == i
    if np.any(sel):
        biny[i] = np.nanmean(flux_all[sel])

good    = ~np.isnan(biny)
centers = centers[good]
biny    = biny[good]

print(f"Rebinned to {len(centers)} channels at R={R}")

## Spectral line catalogue

In [ ]:
lines = [
    # ("[C I] 2-1",          809.41,  2.1),
    ("CH\u207A 1-0",        835.45, -36.1),
    ("CO 7-6",              806.65,  22),
    ("CO 8-7",              921.77,  34),
    ("CO 9-8",             1036.88,  40),
    ("CO 10-9",            1152.47,  53),
    ("\u00B9\u00B3CO 7-6",  771.19,   9.7),
    ("\u00B9\u00B3CO 8-7",  881.23,  12.5),
    ("\u00B9\u00B3CO 9-8",  991.50,  15),
    ("\u00B9\u00B3CO 10-9",1100.62,   8.6),
    ("\u00B9\u00B3CO 11-10",1211.00, 35.8),
    ("HCN 9-8",             797.43,   2.0),
    ("HCN 10-9",            886.84,   3.2),
    ("HCO\u207A 8-7",       713.26,   2.4),
    ("HF 1-0",             1232.86, -179),
    ("H\u2082O 2\u2081,1 - 2\u2080,2",  751.95,   8.5),
    # ("H\u2082O 3\u2081,2 - 3\u2080,3", 1097.38, 25.7),
    ("H\u2082O 1\u2081,1 - 0\u2080,0", 1113.68, -163),
    ("H\u2082O 3\u2082,1 - 3\u2081,2", 1163.11,  40),
    # ("H\u2082O 4\u2082,2 - 4\u2081,3", 1207.42, 30),
    ("H\u2082O 2\u2082,0 - 2\u2081,1", 1228.80,  16.5),
    ("H\u2082O\u207A 1\u2081,1 - 0\u2080,0", 1139.56, -118),
    # ("H\u2082S 2\u2081,2 - 1\u2080,1", 736.35, -9.9),
    ("NH 1\u2080 - 0\u2081",   946.56, -30.0),
    ("NH 1\u2082 - 0\u2081",   974.60, -58.8),
    # ("NH 1\u2082 - 0\u2081", 974.66, -81.7),
    ("NH 1\u2081 - 0\u2081",  1000.15, -87.8),
    ("NH\u2082 2\u2080,2 - 1\u2081,1",  902.42,  -9.6),
    ("NH\u2082 1\u2081,1 - 0\u2080,0",  952.79, -52.0),
    # ("NH\u2082", 959.75, -43.0),
    ("NH\u2082, 1\u2081,1 - 0\u2080,0", 959.68, -73.0),
    ("NH\u2083 2\u2081,-1 - 1\u2081,1", 1215.40, -77.6),
    ("OH\u207A 1\u2080,1 - 0\u2081,2",   909.45, -32.1),
    # ("OH\u207A 1\u2082,2 - 0\u2081,1", 972.0, -104),
    ("OH\u207A 1\u2081,2 - 0\u2081,2",  1033.12, -129),
]

lines_cont = [  # secondary list — placed at different vertical offsets for readability
    ("[C I] 2-1",                        809.41,   2.1),
    ("H\u2082O 3\u2081,2 - 3\u2080,3", 1097.38,  25.7),
    ("H\u2082O 4\u2082,2 - 4\u2081,3", 1207.42,  30),
    ("H\u2082S 2\u2081,2 - 1\u2080,1",  736.35,  -9.9),
    ("NH 1\u2082 - 0\u2081",             974.66, -81.7),
    ("NH\u2082",                          959.75, -43.0),
]

freq_min, freq_max = 714, 1250
filtered_lines = [t for t in lines if freq_min <= t[1] <= freq_max]

## Spectrum plot

In [ ]:
plt.figure(figsize=(14, 8))
plt.plot(freq_all, flux_all, label='Original (Full Resolution)', color='grey', alpha=0.4, lw=0.5)
plt.step(centers, biny, where='mid', lw=1.2, label='Rebinned (R=250)')

plt.title(
    "Vy CMa Spectrum:\n"
    "Approximate RA: 07h 22m 55.30s to 07h 22m 55.34s\n"
    "Approximate Dec: -25d 46\u2019 03.51\u2019\u2019 to -25d 46\u2019 02.59\u2019\u2019"
)
plt.xlabel("Frequency (GHz)")
plt.ylabel("Flux (K)")

ymin, ymax = plt.ylim()
yspan = ymax - ymin

for name, nu, I in lines:
    plt.axvline(nu, color='cornflowerblue', linestyle='--', linewidth=1.0)
    if I > 0:
        plt.text(nu, ymax - 0.05*yspan, name, rotation=90, va='top',    ha='center', fontsize=8)
    elif I < 0:
        plt.text(nu, ymin + 0.05*yspan, name, rotation=90, va='bottom', ha='center', fontsize=8)

for name, nu, I in lines_cont:
    plt.axvline(nu, color='cornflowerblue', linestyle='--', linewidth=1.0)
    if I > 0:
        plt.text(nu, ymax - 0.25*yspan, name, rotation=90, va='top',    ha='center', fontsize=8)
    elif I < 0:
        if name.startswith("H\u2082S"):
            plt.text(nu, ymin + 0.12*yspan, name, rotation=90, va='bottom', ha='center', fontsize=8)
        else:
            plt.text(nu, ymin + 0.25*yspan, name, rotation=90, va='bottom', ha='center', fontsize=8)

# Observation band shading
plt.axvspan(freq_min,          freq_range_max_6, color='hotpink',    alpha=0.1)
plt.axvspan(freq_range_min_3,  freq_range_max_3, color='forestgreen',alpha=0.1)
plt.axvspan(freq_range_min_5,  freq_range_max_5, color='hotpink',    alpha=0.1)
plt.axvspan(freq_range_min_4,  freq_range_max_4, color='forestgreen',alpha=0.1)
plt.axvspan(freq_range_min_2,  freq_range_max_2, color='hotpink',    alpha=0.1)
plt.axvspan(freq_range_min_1,  freq_range_max_1, color='forestgreen',alpha=0.1)
plt.axvspan(freq_range_min_7,  freq_max,          color='hotpink',    alpha=0.1)

plt.legend(loc='lower left')

for label, xpos in zip(["1","2","3","4","5","6","7"],
                        [755, 825, 890, 1010, 1075, 1175, 1230]):
    plt.text(xpos, 2, label, fontsize=14, color='orange')

plt.tight_layout()
plt.show()

## Observation region table

In [ ]:
obs_labels_sorted = ["1","2","3","4","5","6","7"]
obs_ids_sorted    = [obs_ID_6, obs_ID_3, obs_ID_5, obs_ID_4, obs_ID_2, obs_ID_1, obs_ID_7]
ra_vals_sorted    = [RA_6,  RA_3,  RA_5,  RA_4,  RA_2,  RA_1,  RA_7]
dec_vals_sorted   = [Dec_6, Dec_3, Dec_5, Dec_4, Dec_2, Dec_1, Dec_7]

freq_ranges = [
    f"{freq_min:.4f} \u2013 {freq_range_max_6:.4f} GHz",
    f"{freq_range_min_3:.4f} \u2013 {freq_range_max_3:.4f} GHz",
    f"{freq_range_min_5:.4f} \u2013 {freq_range_max_5:.4f} GHz",
    f"{freq_range_min_4:.4f} \u2013 {freq_range_max_4:.4f} GHz",
    f"{freq_range_min_2:.4f} \u2013 {freq_range_max_2:.4f} GHz",
    f"{freq_range_min_1:.4f} \u2013 {freq_range_max_1:.4f} GHz",
    f"{freq_range_min_7:.4f} \u2013 {freq_max:.4f} GHz",
]

table_data = list(zip(obs_labels_sorted, obs_ids_sorted, freq_ranges, ra_vals_sorted, dec_vals_sorted))
columns    = ["Region", "Observation ID", "Frequency Range (GHz)", "RA (J2000)", "Dec (J2000)"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.axis("off")
tbl = ax.table(cellText=table_data, colLabels=columns, loc='center', cellLoc='center')
tbl.set_fontsize(10)
tbl.scale(1.5, 2)
plt.title("HIFI Observation Regions, Frequency Coverage, and Pointing Coordinates")
plt.tight_layout()
plt.show()

## Planck 857 GHz map

Set `PLANCK_FILE` to your local path for `skv1066817696137_1.fits`.

In [ ]:
from astropy.wcs import WCS

# ── Edit this path ───────────────────────────────────────────────────────────
PLANCK_FILE = os.path.join(DATA_DIR, "skv1066817696137_1.fits")
# ─────────────────────────────────────────────────────────────────────────────

planck_hdul = fits.open(PLANCK_FILE)
planck_hdul.info()

In [ ]:
# Inspect header to understand WCS / units
print(repr(planck_hdul[0].header))

In [ ]:
data   = planck_hdul[0].data
header = planck_hdul[0].header
w      = WCS(header)
bunit  = header.get('BUNIT', 'Map units')

print(f"Data shape : {data.shape}")
print(f"Map units  : {bunit}")
print(f"Data range : {np.nanmin(data):.3g} – {np.nanmax(data):.3g}")

## RA/Dec helpers

In [ ]:
def ra_to_deg(ra_string):
    """'07h 22m 58.33s' → decimal degrees."""
    h, m, s = (
        ra_string
        .replace('h', '').replace('m', '').replace('s', '')
        .split()
    )
    return (float(h) + float(m)/60 + float(s)/3600) * 15

def dec_to_deg(dec_string):
    """'-25d 46\' 02.92\'\'  → decimal degrees."""
    parts = (
        dec_string
        .replace('d', ' ').replace("'", ' ').replace('"', ' ')
        .split()
    )
    sign   = -1 if parts[0].startswith('-') else 1
    deg    = abs(float(parts[0]))
    arcmin = float(parts[1])
    arcsec = float(parts[2])
    return sign * (deg + arcmin/60 + arcsec/3600)

## Map plots — wide, zoomed, very zoomed

In [ ]:
colors_map = ['cyan', 'hotpink', 'orange', 'green', 'white', 'yellow', 'red']

ra_deg  = np.array([ra_to_deg(ra)   for ra  in ra_vals_sorted])
dec_deg = np.array([dec_to_deg(dec) for dec in dec_vals_sorted])

xpix, ypix = w.wcs_world2pix(ra_deg, dec_deg, 0)

print("Observation positions:")
for lbl, oid, ra, dec, x, y in zip(obs_labels_sorted, obs_ids_sorted, ra_deg, dec_deg, xpix, ypix):
    print(f"  Region {lbl}, Obs ID {oid}: RA={ra:.6f}, Dec={dec:.6f}, xpix={x:.3f}, ypix={y:.3f}")

print(f"\nPixel span: Δx={np.ptp(xpix):.4f}, Δy={np.ptp(ypix):.4f}")

In [ ]:
pad1, pad2 = 30, 5

xlim_zoom       = (xpix.min() - pad1, xpix.max() + pad1)
ylim_zoom       = (ypix.min() - pad1, ypix.max() + pad1)
xlim_very_zoom  = (xpix.min() - pad2, xpix.max() + pad2)
ylim_very_zoom  = (ypix.min() - pad2, ypix.max() + pad2)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 20),
                                     subplot_kw={'projection': w})

titles = [
    "Planck 857 GHz Survey: I",
    "Planck 857 GHz Survey: I (Zoomed)",
    "Planck 857 GHz Survey: I (Very Zoomed)",
]

for ax, title in zip([ax1, ax2, ax3], titles):
    im = ax.imshow(data, origin='lower', cmap='plasma')
    plt.colorbar(im, ax=ax, label=bunit)
    ax.set_title(title)
    ax.set_xlabel('Right Ascension (J2000)')
    ax.set_ylabel('Declination (J2000)')
    for lbl, oid, ra, dec, color in zip(obs_labels_sorted, obs_ids_sorted, ra_deg, dec_deg, colors_map):
        ax.scatter(
            ra, dec,
            transform=ax.get_transform('world'),
            color=color, edgecolor='black', s=80,
            label=f"Region {lbl}: {oid}"
        )

ax2.set_xlim(xlim_zoom)
ax2.set_ylim(ylim_zoom)
ax3.set_xlim(xlim_very_zoom)
ax3.set_ylim(ylim_very_zoom)
ax3.legend(loc='best', fontsize=8)

plt.tight_layout()
plt.show()